In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from src import config
from src.data_loader import load_images_from_folder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1) Carrega treino e teste
X, y, class_names = load_images_from_folder(config.TRAIN_DIR)
print("Treino (bruto):", X.shape, "| Classes:", class_names)

X_test, y_test, _ = load_images_from_folder(config.TEST_DIR)
print("Teste (bruto):", X_test.shape)

# 2) Separa treino/validacao (80/20, estratificado)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=config.RANDOM_STATE,
)
print("Treino:", X_train.shape, "| Validacao:", X_val.shape)

# 3) Normaliza -- fit SOMENTE no treino
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# 4) PCA -- fit SOMENTE no treino
pca = PCA(n_components=150, random_state=config.RANDOM_STATE)
X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)
X_test = pca.transform(X_test)
print(f"Variancia explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print("Shape final -> treino:", X_train.shape, "| validacao:", X_val.shape, "| teste:", X_test.shape)

# 5) Funcao de avaliacao -- reusada em todos os experimentos
def evaluate_model(model, X, y_true, rotulo):
    y_pred = model.predict(X)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    print(f"  [{rotulo:6s}] Acuracia: {metrics['accuracy']:.4f} | Precisao: {metrics['precision']:.4f} | "
          f"Recall: {metrics['recall']:.4f} | F1: {metrics['f1_score']:.4f}")
    return metrics

results = []

# ============================================================
# Experimento 1: MLP Baseline
# ============================================================
print("\n=== Experimento 1 - MLP Baseline (alpha=0.0001) ===")
mlp_1 = MLPClassifier(
    hidden_layer_sizes=(100,), alpha=0.0001,
    early_stopping=False, max_iter=300, random_state=config.RANDOM_STATE,
)
mlp_1.fit(X_train, y_train)
evaluate_model(mlp_1, X_train, y_train, "TREINO")
metrics_1 = evaluate_model(mlp_1, X_test, y_test, "TESTE")
results.append({"Experimento": "Exp1 - MLP Baseline", **metrics_1})

# ============================================================
# Experimento 2: MLP + L2 forte (unica mudanca: alpha 0.0001 -> 1.0)
# ============================================================
print("\n=== Experimento 2 - MLP + L2 forte (alpha=1.0) ===")
mlp_2 = MLPClassifier(
    hidden_layer_sizes=(100,), alpha=1.0,
    early_stopping=False, max_iter=300, random_state=config.RANDOM_STATE,
)
mlp_2.fit(X_train, y_train)
evaluate_model(mlp_2, X_train, y_train, "TREINO")
metrics_2 = evaluate_model(mlp_2, X_test, y_test, "TESTE")
results.append({"Experimento": "Exp2 - MLP + L2 forte", **metrics_2})

# ============================================================
# Experimento 3: MLP + Early Stopping (volta ao alpha padrao, muda so o early_stopping)
# ============================================================
print("\n=== Experimento 3 - MLP + Early Stopping ===")
mlp_3 = MLPClassifier(
    hidden_layer_sizes=(100,), alpha=0.0001,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=10,
    max_iter=300, random_state=config.RANDOM_STATE,
)
mlp_3.fit(X_train, y_train)
print(f"  Parou na iteracao {mlp_3.n_iter_}  (limite maximo era 300)")
evaluate_model(mlp_3, X_train, y_train, "TREINO")
metrics_3 = evaluate_model(mlp_3, X_test, y_test, "TESTE")
results.append({"Experimento": "Exp3 - MLP + Early Stopping", **metrics_3})

# ============================================================
# Tabela parcial
# ============================================================
print("\n=== Resultados acumulados ===")
print(pd.DataFrame(results))

Carregando 'glioma' (1400 imagens)...
  ... 200/1400 (0.1s decorridos)
  ... 400/1400 (0.3s decorridos)
  ... 600/1400 (0.4s decorridos)
  ... 800/1400 (0.6s decorridos)
  ... 1000/1400 (0.7s decorridos)
  ... 1200/1400 (0.9s decorridos)
  ... 1400/1400 (1.0s decorridos)
  'glioma' concluida em 1.0s
Carregando 'meningioma' (1400 imagens)...
  ... 200/1400 (0.2s decorridos)
  ... 400/1400 (0.3s decorridos)
  ... 600/1400 (0.5s decorridos)
  ... 800/1400 (0.7s decorridos)
  ... 1000/1400 (0.9s decorridos)
  ... 1200/1400 (1.0s decorridos)
  ... 1400/1400 (1.2s decorridos)
  'meningioma' concluida em 1.2s
Carregando 'notumor' (1400 imagens)...
  ... 200/1400 (0.1s decorridos)
  ... 400/1400 (0.3s decorridos)
  ... 600/1400 (0.5s decorridos)
  ... 800/1400 (0.6s decorridos)
  ... 1000/1400 (0.8s decorridos)
  ... 1200/1400 (0.9s decorridos)
  ... 1400/1400 (1.0s decorridos)
  'notumor' concluida em 1.0s
Carregando 'pituitary' (1400 imagens)...
  ... 200/1400 (0.2s decorridos)
  ... 400/140